# Round 0 · Explore the log (25 min)

Ninety days of a store's account audit log: about 335,000 events, 11,000 accounts, no labels. Somewhere in it are about ninety confirmed account takeovers in eight campaigns. Before you build a single detection, learn what normal looks like, and find the data-quality traps before they find you.

**Deliverable:** three written hypotheses at the bottom of this notebook. Nothing to submit yet.

If `load_events()` fails, generate the data first: `python tools/make_log.py` (about a minute).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../tools"))
from atolab import *
import matplotlib.pyplot as plt
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

df = load_events()            # ~10 s. Deliberately keeps duplicate and retried lines.
print(len(df), "events,", df.account_id.nunique(), "accounts,", df.ts.min(), "->", df.ts.max())
df.head(3)

## 0.1 What is in it

Event mix, outcomes, who acts (`actor`), how people log in (`auth_method`). Note the share of `session_resume`: most logins are a browser resuming a remembered session, not a password.

In [ ]:
print(df.event_type.value_counts().to_string())

In [ ]:
logins = df[df.event_type == "login_success"]
print(logins.auth_method.value_counts(normalize=True).round(3).to_string())
print(); print(df.actor.value_counts().to_string())
print(); print(df.asn_type.value_counts().to_string())

In [ ]:
ax = df.set_index("ts").resample("D").size().plot(figsize=(12, 3), title="events per day")
ax.set_ylabel("events"); plt.show()

## 0.2 The traps

Four things are wrong with this log on purpose, because they are wrong with every real one. Find each before reading the next cell's comment.

1. Some lines appear twice. 2. Some requests were retried. 3. Some events were ingested hours after they happened, and a few *before* (clock skew). 4. One block of addresses changes country for three days without changing anything else.

In [ ]:
dup_lines = df.event_id.duplicated().sum()
retries = df.request_id.duplicated().sum() - dup_lines
lag = (df.ingested_at - df.ts).dt.total_seconds()
print(f"duplicate lines: {dup_lines}   retries: {retries}   late (>30 min): {(lag > 1800).sum()}   clock ahead of ingestion: {(lag < 0).sum()}")

In [ ]:
# TODO: find the address block whose country changes for three days. Hint: group by the /24 of `ip`
# (first three octets) and count distinct countries; then look at when.
blk = df[df.ip.str.contains(r"^\d+\.")].copy()
blk["block"] = blk.ip.str.rsplit(".", n=1).str[0]
multi = blk.groupby("block").country.nunique()
print(multi[multi > 1].head())

## 0.3 Timelines

`timeline(df, account_id)` prints one account with the gap to the previous event and novelty flags (`NEW-DEV`, `NEW-CTRY`, `NEW-ASN`, `NEW-JA4`) computed against *that account's* earlier events only. Look at a few: an active shopper, a quiet one, one with a `password_reset_completed`, one with a `support_reset_issued`. What does "normal" look like for each?

In [ ]:
timeline(df, "acc_000042")

In [ ]:
# TODO: pick accounts by a property. Example: accounts with a support reset
resets = df[df.event_type == "support_reset_issued"].account_id.unique()
print(len(resets), "accounts had a support reset")
timeline(df, resets[0], around=df[(df.account_id == resets[0]) & (df.event_type == "support_reset_issued")].event_id.iloc[0], n=20)

## 0.4 Clean up and move on

Now that you have *seen* the duplicates and retries, drop them. Every later notebook starts with `dedupe(load_events())`.

In [ ]:
df = dedupe(df)

## 0.5 Three hypotheses

Write them here, as sentences with numbers: *"A login from a country the account has never used, followed within an hour by an email change, is a takeover; I expect fewer than 50 such accounts."*

1. …
2. …
3. …

Then open `01_rules.ipynb`.